# # Load training data

In [ ]:
from src.demo import prepare
conn_name = prepare()

from nubison_model import data

db = data.connection(conn_name)
df = db.load("SELECT * FROM iris")
datasets = data.split(df, {"train": 0.8, "val": 0.2}, random_state=42)

for name, sub in datasets.items():
    print(f"{name:6s} rows={len(sub):3d}")


# # Train (PyTorch Lightning)

In [ ]:
# `train(datasets, target, *, ...)` parameters:
#   datasets      — dict from `data.split` (must include "train";
#                   "val" enables `t.X_val / t.y_val` + auto-scoring;
#                   "test" populates `t.X_test / t.y_test`)
#   target        — label column name(s); single string or list of strings.
#                   Splits each split's DataFrame into X = features and
#                   y = target so estimators can call `fit(X, y)`.
#   model_type    — free-form string tagged on the run as `model_type`
#                   (surfaced in the nubison UI). "classifier" and
#                   "regressor" additionally make `t.fit()` log
#                   `val_accuracy` / `val_r2`. None skips the tag.
#   weights_path  — where `t.save(model)` writes the pickle.
#                   Default "src/weights.pkl" matches `register(artifact_dirs="src")`.
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import pytorch_lightning as pl
from nubison_model import train


class IrisLightning(pl.LightningModule):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 3))
        self.criterion = nn.CrossEntropyLoss()

    def forward(self, x):
        return self.fc(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        loss = self.criterion(self(x), y)
        self.log("loss", loss, prog_bar=False)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=0.01)


with train(datasets=datasets, target=["target"], model_type="classifier") as t:
    X = torch.tensor(t.X_train.values, dtype=torch.float32)
    y = torch.tensor(t.y_train.values.ravel(), dtype=torch.long)
    loader = DataLoader(TensorDataset(X, y), batch_size=16, shuffle=True)

    model = IrisLightning()
    trainer = pl.Trainer(
        max_epochs=30,
        accelerator="cpu",
        enable_progress_bar=False,
        logger=False,
        enable_checkpointing=False,
    )
    trainer.fit(model, loader)

    t.save(model)

print(f"run_id: {t.run_id}")
